# 04 — Read the retained paper results

    This notebook fits nothing. It reads the original CSV outputs, confirms the
    principal reported comparisons and plots the trade-offs. Percentages are
    computed from fractional rate columns; subgroup and aggregate denominators
    are deliberately kept distinct.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from threadpoolctl import threadpool_limits

# The same notebook works from the repository root or notebooks/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").exists() and (p / "src/attendpro_paper2").exists())
thread_limit = threadpool_limits(limits=1)
pd.set_option("display.max_columns", 12)

In [ ]:
tables = ROOT / "results/paper"
nominal = pd.read_csv(tables / "extension_mean_metrics.csv")
nominal = nominal[(nominal.condition == "nominal") & (nominal.operating_point == "FAR_1pct")]
order = ["Tuned hard rule", "Logistic additive", "Logistic interactions", "Logistic no reliability",
         "GAM additive", "Random forest", "Gradient boosting"]
nominal = nominal.set_index("method").loc[order]
display(nominal[["FAR", "FRR", "BA"]].mul(100).round(2))
assert round(100*nominal.loc["Gradient boosting", "FAR"], 2) == .55
assert round(100*nominal.loc["Tuned hard rule", "FRR"], 2) == 34.95

## Boundary errors hidden by the nominal averages

    Frozen original, cutoff-only adaptation and complete redevelopment are compared
    on the same near-boundary test population. Genuine outsiders carry correct
    identity and live evidence but fail the location authorization condition.

In [ ]:
boundary = pd.read_csv(tables / "near_boundary_key_results.csv")
boosting = boundary[boundary.method == "Gradient boosting"].set_index("arm")
arms = ["Frozen original", "Threshold adapted", "Boundary trained"]
boosting = boosting.loc[arms]
display(boosting[["FAR", "FRR", "genuine_outside", "boundary_inside_rejection"]].mul(100).round(2))
ax = boosting[["genuine_outside", "boundary_inside_rejection"]].mul(100).plot.bar(figsize=(8, 4))
ax.set(ylabel="Subgroup error (%)", xlabel="", title="Boundary trade-off in retained test results")
ax.tick_params(axis="x", rotation=0)
ax.legend(["Genuine outsiders accepted", "Boundary insiders rejected"])
plt.tight_layout()
plt.show()

In [ ]:
dependence = pd.read_csv(tables / "dependence_mean_metrics.csv")
selected = dependence[(dependence.population == "nominal") &
                       dependence.method.isin(["Gradient boosting", "Tuned hard rule"])]
display(selected.pivot(index=["config", "protocol"], columns="method", values=["FAR", "FRR"])
        .mul(100).round(2))

## Gate floor and paired fit ranges

    These describe the original nominal test draws and three development fits.
    Fit ranges are not confidence intervals. The gate floor averages rates within
    fit and then across fits; it is not fitted or tuned after viewing the tests.

In [ ]:
gates = pd.read_csv(tables / "reviewer_gate_diagnostics.csv")
paired = pd.read_csv(tables / "reviewer_paired_fit_differences.csv")
gate_fit_means = gates.groupby("fit").gate_frr.mean().mul(100)
print("Common gate floor (%):", round(gate_fit_means.mean(), 2))
display(gate_fit_means.rename("legitimate_gate_rejection_percent").to_frame())
display(paired)
assert round(gate_fit_means.mean(), 2) == 17.09

### Continue the research

The light package retains only selected result tables. Run scripts/reproduce.py
in a new output folder to generate all models and full batch-level results.
Its workspace contains extension/replicate_metrics.csv, boundary_training/metrics.csv
and dependence/batch_metrics.csv. The README explains the workflow and denominators.